# 🔬 MorphoBricks preprocessing

Prepare microscopy TIFF or OME-Zarr experiments for the other MorphoBricks bricks. The notebook never changes the source images; results are written to a separate output folder.

### What it does

- **Checks the experiment folder and metadata** and reports problems before pixels are processed.
- **Confirms the experimental design**: folder labels, biological-repeat pairing and controls.
- **Confirms image interpretation**: axis roles and fluorescence/RGB image type. Scientific metadata are never silently guessed.
- **Computes per-plane intensity histograms** used by downstream MorphoBricks.
- **Runs intensity QC** and, on request, compares groups after preprocessing.

### Before you start

- Put the experiment in one folder containing TIFF (.tif/.tiff) or OME-Zarr images.
- Optional masks can be supplied from a separate masks folder.
- **Google Colab:** keep the experiment and output on Google Drive, under `/content/drive/MyDrive/`.
- **JupyterLab / LabConstrictor:** use normal local paths.

### How to run

Run the three code cells below in order, from top to bottom (click a cell and press **Shift + Enter**, or its ▶ button):

1. **Install** prepares the environment.
2. **Mount Google Drive** connects your Drive on Google Colab; elsewhere it does nothing.
3. **Run preprocessing** opens the complete preprocessing app.

After a restart of the runtime or kernel, run the three cells again: cell 1 then installs nothing that is already there.

___

## **1. Install MorphoBricks preprocessing**

Installs the preprocessing release this notebook is made for, with its notebook dependencies (on Google Colab also the OME-Zarr reader), into the Python environment of this notebook. Nothing is installed when it is already there, so running the cell again is safe.

When it finishes, it prints **Ready** with the versions of preprocessing, normalization and ipywidgets, the notebook version and where the package was loaded from. No restart of the runtime is needed.

In [ ]:
# @title 1. Install MorphoBricks preprocessing
import platform
import subprocess
import sys
import tomllib
from importlib import import_module, invalidate_caches
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from pathlib import Path

# LabConstrictor version identifiers.
current_version = "0.5.0"
notebook_name = "MorphoBricks_Preprocessing"

# A downloaded/Colab notebook installs this immutable source revision.
RELEASE_TAG = "v0.5.0"
RELEASE_REF = "11b627edee73a1dba2d6aacf1940e9081d642ad6"
REPOSITORY = "https://github.com/CellMigrationLab/morphobricks-preprocessing.git"
EXTRAS = "notebook"

# A nearby local checkout is used only when it is exactly the RELEASE_REF commit with no
# uncommitted changes. Set True only to develop against another commit or local edits: the
# printed header then says so, and the checkout's commit and tree state are shown either way.
DEVELOPMENT_OVERRIDE = False
SETUP_RECORD: dict = {}

if "google.colab" in sys.modules:
    print("Google Colab: preparing MorphoBricks preprocessing...", flush=True)
    EXTRAS = "notebook,zarr"
else:
    print(
        f"JupyterLab: Python {platform.python_version()} on {platform.platform()}\n"
        f"Working folder: {Path.cwd()}",
        flush=True,
    )


def _checkout_version(checkout: Path) -> str | None:
    """Return the nearby checkout package version, or None when it cannot be read."""
    try:
        data = tomllib.loads((checkout / "pyproject.toml").read_text(encoding="utf-8"))
    except (OSError, tomllib.TOMLDecodeError):
        return None
    value = data.get("project", {}).get("version")
    return str(value) if value is not None else None


def _git(checkout: Path, *arguments: str) -> str | None:
    """Return the output of ``git -C checkout ...``, or None when Git cannot answer."""
    try:
        return subprocess.check_output(
            ["git", "-C", str(checkout), *arguments], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (subprocess.CalledProcessError, OSError):
        return None


def _checkout_identity(checkout: Path) -> tuple[str | None, bool | None]:
    """Return the checkout's Git commit and whether its tree has changes (None when unknown).

    Unknown means not a Git work tree rooted at the checkout, or Git is not available: the code
    identity cannot then be shown, and it is never guessed.
    """
    top = _git(checkout, "rev-parse", "--show-toplevel")
    if top is None or Path(top).resolve() != checkout.resolve():
        return None, None
    commit = _git(checkout, "rev-parse", "HEAD")
    status = _git(checkout, "status", "--porcelain")
    return commit, (None if status is None else bool(status))


def _require_release_checkout(checkout: Path) -> tuple[str | None, bool | None]:
    """Show the checkout's commit and tree state; refuse one that is not the clean release."""
    commit, dirty = _checkout_identity(checkout)
    differences = []
    if commit != RELEASE_REF:
        differences.append(
            f"its commit is {commit or 'unknown (not a Git work tree, or Git is unavailable)'}, "
            f"not RELEASE_REF {RELEASE_REF}"
        )
    if dirty is not False:
        differences.append(
            "it has uncommitted changes" if dirty else "its tree state could not be read"
        )
    if differences and not DEVELOPMENT_OVERRIDE:
        raise RuntimeError(
            f"The local checkout {checkout} is not the release this notebook pins: "
            + "; ".join(differences)
            + f". Check out {RELEASE_REF} with a clean tree, or set DEVELOPMENT_OVERRIDE = True "
            "at the top of this cell to develop against this checkout, then run the cell again."
        )
    print(f"Local checkout {checkout}: {_describe_checkout(commit, dirty)}", flush=True)
    if DEVELOPMENT_OVERRIDE:
        print(
            "DEVELOPMENT OVERRIDE is on: this is not necessarily the release "
            f"{RELEASE_TAG} ({RELEASE_REF[:12]}).",
            flush=True,
        )
    return commit, dirty


def _record_setup(
    origin: str, package_version: str, checkout: Path | None, commit: str | None, dirty: bool | None
) -> str:
    """Record what this kernel loaded in SETUP_RECORD and return the header's checkout line."""
    SETUP_RECORD.clear()
    SETUP_RECORD.update(
        notebook_version=current_version,
        preprocessing_version=package_version,
        source=origin,
        release_ref=RELEASE_REF,
        checkout_commit=commit,
        checkout_has_uncommitted_changes=dirty,
        development_override=DEVELOPMENT_OVERRIDE,
    )
    if checkout is None:
        return ""
    override = "; DEVELOPMENT OVERRIDE on" if DEVELOPMENT_OVERRIDE else ""
    return f"Checkout: {_describe_checkout(commit, dirty)}{override}.\n"


def _describe_checkout(commit: str | None, dirty: bool | None) -> str:
    """Return the checkout's commit and tree state as one line."""
    state = {True: "uncommitted changes", False: "clean tree", None: "tree state unknown"}[dirty]
    return f"commit {commit or 'unknown'}, {state}"


def _missing_requirements() -> list[str]:
    """Return notebook requirements this kernel cannot import."""
    required = (
        "morphobricks_preprocessing",
        "morphobricks_normalization",
        "numpy",
        "tifffile",
        "ipywidgets",
        "matplotlib",
        *(("zarr",) if "zarr" in EXTRAS.split(",") else ()),
    )
    return [name for name in required if find_spec(name) is None]


def _install_preprocessing() -> None:
    """Prepare this kernel: install what is missing, check the interface loads, print versions.

    Running it again installs nothing that is already there.
    """
    if sys.version_info < (3, 11):
        raise RuntimeError("Select a Python 3.11 or newer kernel, then run this cell again.")

    current = Path.cwd().resolve()
    nearby = next(
        (
            folder
            for folder in (current, *current.parents)
            if (folder / "pyproject.toml").is_file()
            and (folder / "src/morphobricks_preprocessing/__init__.py").is_file()
        ),
        None,
    )
    checkout = (
        nearby if nearby is not None and _checkout_version(nearby) == current_version else None
    )

    commit, dirty = _require_release_checkout(checkout) if checkout is not None else (None, None)
    if checkout is not None:
        source = checkout / "src"
        loaded = sys.modules.get("morphobricks_preprocessing")
        if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(source):
            raise RuntimeError(
                "This kernel already loaded another MorphoBricks preprocessing copy. "
                "Restart the kernel, then run this cell again."
            )
        sys.path[:] = [str(source), *(entry for entry in sys.path if entry != str(source))]
        invalidate_caches()
        install_target = ["-e", f"{checkout}[{EXTRAS}]"]
        origin = f"local checkout {checkout}"
    else:
        install_target = [f"morphobricks-preprocessing[{EXTRAS}] @ git+{REPOSITORY}@{RELEASE_REF}"]
        try:
            installed = version("morphobricks-preprocessing")
        except PackageNotFoundError:
            installed = None
        origin = (
            f"installed package {installed}"
            if installed == current_version
            else f"pinned {RELEASE_TAG} candidate ({RELEASE_REF[:12]})"
        )

    missing = _missing_requirements()
    try:
        installed = version("morphobricks-preprocessing")
    except PackageNotFoundError:
        installed = None
    if checkout is None and installed != current_version:
        missing.append(f"morphobricks-preprocessing {current_version}")

    if missing:
        print(
            "Installing notebook requirements (" + ", ".join(dict.fromkeys(missing)) + ")...",
            flush=True,
        )
        try:
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "--disable-pip-version-check",
                    *install_target,
                ],
                check=True,
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            raise RuntimeError(
                "Notebook setup could not install its requirements. Check the installation "
                "output and internet access, then run this cell again."
            ) from exc
        invalidate_caches()

    remaining = _missing_requirements()
    if remaining:
        raise RuntimeError(
            "Installation finished, but this kernel still cannot import "
            + ", ".join(remaining)
            + ". Restart the kernel and run this cell again."
        )

    try:
        import_module("morphobricks_preprocessing.notebook_ui")
    except ImportError as exc:
        raise RuntimeError(
            "MorphoBricks preprocessing was found, but its notebook interface could not load."
        ) from exc

    package_version = version("morphobricks-preprocessing")
    normalization_version = version("morphobricks-normalization")
    widget_version = version("ipywidgets")
    package_file = Path(import_module("morphobricks_preprocessing").__file__).resolve()
    checkout_line = _record_setup(origin, package_version, checkout, commit, dirty)
    print(
        f"Ready: preprocessing {package_version}; normalization {normalization_version}; "
        f"ipywidgets {widget_version}.\n"
        f"Notebook {current_version}; source: {origin}.\n"
        f"{checkout_line}"
        f"Loaded from: {package_file}\n"
        "Next: run cell 2 (Mount Google Drive), then cell 3 (Run preprocessing).",
        flush=True,
    )


_install_preprocessing()

___

## **2. Mount Google Drive**

**Google Colab:** connects your Google Drive. Allow access in the window that opens; your Drive is then the folder **/content/drive/MyDrive**. Keep the experiment and the output folder there: everything else on a Colab runtime is wiped when it disconnects.

**JupyterLab / LabConstrictor:** there is no Drive to mount. The cell only prints a note, and you use normal local paths in the app.

In [ ]:
# @title 2. Mount Google Drive
import sys

if "google.colab" in sys.modules:
    from google.colab import drive

    print("Google Colab: connecting Google Drive...", flush=True)
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        raise RuntimeError(
            "Google Drive could not be connected. Allow Drive access, then run this cell again."
        ) from exc
    print("Google Drive is connected: your files are in /content/drive/MyDrive.", flush=True)
else:
    print("Not Google Colab: there is no Google Drive to mount here, so nothing was done.")

___

## **3. Run preprocessing**

Opens the preprocessing app. Run cells 1 and 2 first: without them this cell stops and says which cell to run.

In the app:

1. Give the **experiment folder** (masks and output folders are optional) and click **Scan the folders**. Look at the folders and the example image, then click **Looks right: read all image headers**.
2. Go through the steps one at a time with **Next** (or click a step in the strip at the top): axes and pixel size, conditions, replicates, comparisons, image type, stains, channel tags and masks.
3. On **Check and save**, read your experiment in words and save it; then **Run** computes the histograms.
4. After the run, the intensity checks and **Compare groups** are collapsed panels of the run step.

Every wait shows its progress, and a greyed-out button says beside it what it waits for.

In [ ]:
# @title 3. Run preprocessing
import importlib
import sys
from pathlib import Path

if not globals().get("SETUP_RECORD"):
    raise RuntimeError(
        "MorphoBricks preprocessing is not installed in this session yet: run cell 1 "
        "(Install MorphoBricks preprocessing) first, then run this cell again."
    )
if "google.colab" in sys.modules:
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError(
            "Google Drive is not connected, so the experiment and output folders on Drive "
            "cannot be reached: run cell 2 (Mount Google Drive) first, then run this cell again."
        )

try:
    interface = importlib.import_module("morphobricks_preprocessing.notebook_ui")
except ImportError as exc:
    raise RuntimeError(
        "MorphoBricks preprocessing was found, but its notebook interface could not load. "
        "Run cell 1 again."
    ) from exc
notebook = interface.PreprocessingNotebook()

import ipywidgets as widgets
from IPython.display import display

app = widgets.VBox(
    [notebook.welcome(), notebook.steps(), notebook.results_step()],
    layout=widgets.Layout(width="99%"),
)
display(app)